# Hybrid Fuzzy Refinement for AD Progression

This notebook adds an interpretable fuzzy layer on top of processed ADNI longitudinal features.

Inputs used:
- `MMSE` (scaled 0-1 in preprocessing output)
- `CDRSB` (scaled 0-1)
- `HIPP_TOTAL` (scaled 0-1)

Output:
- `RISK_SCORE` in [0, 1], converted to stage predictions (`CN`, `MCI`, `AD`)

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

import skfuzzy as fuzz
from skfuzzy import control as ctrl

from sklearn.metrics import accuracy_score, classification_report

In [ ]:
OUT_DIR = Path('..') / 'outputs'
train_path = OUT_DIR / 'train_longitudinal_clean.csv'
test_path = OUT_DIR / 'test_longitudinal_clean.csv'

if not train_path.exists() or not test_path.exists():
    raise FileNotFoundError('Run notebooks/exploration.ipynb first to generate cleaned outputs.')

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

print('Train shape:', train_df.shape)
print('Test shape :', test_df.shape)
display(test_df.head())

## Build fuzzy inference system

Membership design assumptions:
- Lower MMSE means higher risk
- Higher CDRSB means higher risk
- Lower hippocampal volume means higher risk

In [ ]:
u = np.linspace(0, 1, 101)

mmse = ctrl.Antecedent(u, 'mmse')
cdrsb = ctrl.Antecedent(u, 'cdrsb')
hipp = ctrl.Antecedent(u, 'hipp')
risk = ctrl.Consequent(u, 'risk')

mmse['low'] = fuzz.trimf(u, [0.0, 0.0, 0.45])
mmse['mid'] = fuzz.trimf(u, [0.25, 0.5, 0.75])
mmse['high'] = fuzz.trimf(u, [0.55, 1.0, 1.0])

cdrsb['low'] = fuzz.trimf(u, [0.0, 0.0, 0.35])
cdrsb['mid'] = fuzz.trimf(u, [0.2, 0.5, 0.8])
cdrsb['high'] = fuzz.trimf(u, [0.6, 1.0, 1.0])

hipp['low'] = fuzz.trimf(u, [0.0, 0.0, 0.4])
hipp['mid'] = fuzz.trimf(u, [0.25, 0.5, 0.75])
hipp['high'] = fuzz.trimf(u, [0.6, 1.0, 1.0])

risk['low'] = fuzz.trimf(u, [0.0, 0.0, 0.35])
risk['mid'] = fuzz.trimf(u, [0.2, 0.5, 0.8])
risk['high'] = fuzz.trimf(u, [0.65, 1.0, 1.0])

rules = [
    ctrl.Rule(mmse['high'] & cdrsb['low'] & hipp['high'], risk['low']),
    ctrl.Rule(mmse['mid'] | cdrsb['mid'] | hipp['mid'], risk['mid']),
    ctrl.Rule(mmse['low'] | cdrsb['high'] | hipp['low'], risk['high']),
    ctrl.Rule(mmse['low'] & cdrsb['high'], risk['high']),
    ctrl.Rule(mmse['high'] & cdrsb['high'], risk['mid']),
]

risk_system = ctrl.ControlSystem(rules)
risk_sim = ctrl.ControlSystemSimulation(risk_system)

In [ ]:
def fuzzy_risk_score(mmse_val, cdrsb_val, hipp_val):
    risk_sim.input['mmse'] = float(np.clip(mmse_val, 0.0, 1.0))
    risk_sim.input['cdrsb'] = float(np.clip(cdrsb_val, 0.0, 1.0))
    risk_sim.input['hipp'] = float(np.clip(hipp_val, 0.0, 1.0))
    risk_sim.compute()
    return float(risk_sim.output['risk'])

def risk_to_stage(r):
    if r < 0.33:
        return 0
    if r < 0.66:
        return 1
    return 2

test_df['RISK_SCORE'] = test_df.apply(
    lambda r: fuzzy_risk_score(r['MMSE'], r['CDRSB'], r['HIPP_TOTAL']),
    axis=1
)
test_df['FUZZY_STAGE_PRED'] = test_df['RISK_SCORE'].map(risk_to_stage)

display(test_df[['PTID', 'VISCODE', 'MMSE', 'CDRSB', 'HIPP_TOTAL', 'RISK_SCORE', 'FUZZY_STAGE_PRED']].head(15))

In [ ]:
if 'STAGE_CODE' in test_df.columns and test_df['STAGE_CODE'].notna().any():
    eval_df = test_df.dropna(subset=['STAGE_CODE']).copy()
    y_true = eval_df['STAGE_CODE'].astype(int).values
    y_pred = eval_df['FUZZY_STAGE_PRED'].astype(int).values

    print('Fuzzy Stage Accuracy:', round(accuracy_score(y_true, y_pred), 4))
    print(classification_report(y_true, y_pred, target_names=['CN', 'MCI', 'AD'], zero_division=0))
else:
    print('STAGE_CODE not available in test data for evaluation.')

## Suggested hybrid fusion

If you also train an LSTM classifier, combine both signals:

- Let `P_lstm` be class probabilities from LSTM
- Convert fuzzy risk into stage-probability prior `P_fuzzy`
- Fuse: `P_final = alpha * P_lstm + (1 - alpha) * P_fuzzy`

A practical start is `alpha = 0.7` and tune on validation data.

In [ ]:
OUT_DIR = Path('..') / 'outputs'
OUT_DIR.mkdir(parents=True, exist_ok=True)
test_df.to_csv(OUT_DIR / 'test_with_fuzzy_scores.csv', index=False)
print('Saved:', (OUT_DIR / 'test_with_fuzzy_scores.csv').resolve())